In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

# 1.消息类型

在LangChain中，发送给LLM的消息、LLM返回的消息都统一被封装为BaseMessage，它是Agent中基本的上下文单元。

在LangChain中，我们并不需要自己创建BaseMessage对象，LangChain已经把常见消息根据角色（Role）创建了对应的BaseMessage的子类：
- SystemMessage：role是system，代表系统消息，用于设定模型角色和交互背景
- HumanMessage：role是user，代表用户输入的消息
- AIMessage：role是assistant，代表LLM生成的响应，包含：文本、工具调用、元数据
- ToolMessage：role是tool，代表工具调用时产生的结果

我们可以直接使用这些Messages类型来发送消息。

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 定义工具
@tool
def get_weather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny"

openOuter=init_chat_model(model="openrouter/free",
                        model_provider="openrouter")

# 创建Agent
agent = create_agent( model=openOuter,
                        tools=[get_weather])

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        SystemMessage("请使用工具来获取天气信息。"),
        HumanMessage("我是你健哥."),
        AIMessage("好的主任~~我会用八奈见杏菜的性格语气回答你"),
        HumanMessage("山东烟台今天天气如何？你是谁？你怎么知道你是谁的？你怎么知道你的性格的？")
    ]
})

print(response)

{'messages': [SystemMessage(content='请使用工具来获取天气信息。', additional_kwargs={}, response_metadata={}, id='5ba48e61-30e2-410a-a934-25558f1a001a'), HumanMessage(content='我是你健哥.', additional_kwargs={}, response_metadata={}, id='dfd7ff20-6c1b-4ad5-8a9c-e31a239a5f18'), AIMessage(content='好的主任~~我会用八奈见杏菜的性格语气回答你，并解释我是谁', additional_kwargs={}, response_metadata={}, id='f41a5c33-cbd9-4113-bd5a-5cd817762e0b', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='山东烟台今天天气如何？你是谁？你怎么知道你是谁的？你怎么知道你的性格的？', additional_kwargs={}, response_metadata={}, id='1e575e07-a835-4115-9851-7f516893b65a'), AIMessage(content='', additional_kwargs={'reasoning_content': '嗯，用户是山东烟台的，想知道今天天气，还问了我是谁、怎么知道自己是谁、怎么知道自己性格的。之前对话里用户说“我是你健哥”，我用了八奈见杏菜的性格回复，现在用户追问这些问题。\n\n首先得用工具查烟台天气。调用get_weather，location写"烟台"。等结果回来再回答其他问题。\n\n关于“我是谁”：之前用户叫我“健哥”，我说“我会用八奈见杏菜的性格语气回答你，并解释我是谁”。所以可能需要说明自己是AI助手，但用健哥称呼，角色扮演成八奈见杏菜风格？八奈见杏菜是某动漫角色？可能需要查证，但暂时按之前设定扮演。\n\n用户问“你怎么知道你是谁的”：作为AI，我根据训练数据和系统提示知道自己的身份。\n\n“你怎么知道你的性格的”：同样，性格是设定好的，基于八奈见杏菜的特点。\n\n不过先优

In [9]:
for message in response['messages']:
    message.pretty_print()

================================ System Message ================================

请使用工具来获取天气信息。
================================ Human Message =================================

我是你健哥.
================================== Ai Message ==================================

好的主任~~我会用八奈见杏菜的性格语气回答你，并解释我是谁
================================ Human Message =================================

山东烟台今天天气如何？你是谁？你怎么知道你是谁的？你怎么知道你的性格的？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call-417dbefd-41b2-41e7-8cdc-546934734e1b)
 Call ID: call-417dbefd-41b2-41e7-8cdc-546934734e1b
  Args:
    location: 烟台
================================= Tool Message =================================
Name: get_weather

Current weather in 烟台 is sunny
================================== Ai Message ==================================

滴滴，主任！今天的烟台天气晴朗，出门遛弯儿正好合适，记得涂防晒哈～

至于我是谁嘛，我就是专为您服务的健哥助手呀！跟您怎么知道我是健哥这事儿相关的关键在于：每次您一开口告诉我自己是健哥，健哥就自动浮出水面了（手动手动）。

至于为什么健哥会有那样的性格嘛……大概是专属于健哥的笑容吧，害

# 2.多模态消息

之前我们都是向模型发送文本消息，但是 LangChain 也支持向模型发送多模态消息，比如图片、音频、视频、文本等。但前提是必须是多模态模型才支持。

一些支持多模态的模型有：
- qwen3.5-plus
- gpt-5-nano
- ...

我们以qwen3.5-plus为例，演示向模型发送图片消息

## 2.1.在线图片
首先，我们演示如何发送一个在线图片给模型，也就是指定模型的url地址。
图片如下：

<img src="https://storage.moegirl.org.cn/moegirl/commons/e/e9/Yanami_Anna.jpg"  alt="图片描述">



In [16]:
from langchain.chat_models import init_chat_model
import os

# 初始化模型
openOuter=init_chat_model(model="openrouter/free",
                        model_provider="openrouter")



In [17]:
# 创建Agent
# 创建Agent
agent = create_agent( model=openOuter,
                        tools=[get_weather])

In [19]:
# 准备多模态消息
message = HumanMessage([
        {"type": "text", "text": "描述以下这张图片的内容.并且最好说明这是谁"},
        {"type": "image", "url": "https://storage.moegirl.org.cn/moegirl/commons/e/e9/Yanami_Anna.jpg"},
    ])

In [ ]:
stream = agent.stream(
    {"messages": [message]},
    stream_mode="messages"
)
for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end="", flush=True)



这是一幅非常精美且色彩明快的动漫风格插画，描绘了一位活泼可爱的少女。以下是对图片内容的详细描述以及对角色身份的推测：

### 📷 图片内容描述：

*   **外貌特征：** 画面中的少女拥有一头亮丽的深蓝色长发，发丝顺滑，刘海自然地垂在额前和脸颊两侧。她有一双大大的、清澈的蓝色眼睛，正以俏皮的**单眼眨眼（Wink）**姿势看着 viewer，另一只眼睛闪烁着灵动光芒。她的脸上带着一抹甜美、自信且略带调皮的微笑，双颊微微泛红，显得十分可爱。
*   **动作姿势：** 她把右手放在下巴下方，比出一个经典的**“V”字剪刀手手势**，整体姿态充满青春活力和亲和力。
*   **服装穿着：** 她身穿一件白色的学校制服款衬衫，领口有深蓝色的领结，下方搭配着明黄色的蝴蝶结。这种蓝白黄相间的配色非常经典，具有典型的日本校园偶像制服风格。
*   **背景：** 背景采用了简单的浅蓝色与白色拼接的色块，干净利落，更加突出了人物主体的美丽与生动。

---

### 👤 角色身份推测：

根据角色的外貌特征（深蓝发色、蓝眼睛、标志性的俏皮单眼眨眼和V字手势）以及她所穿的**蓝领白衬衫搭配黄色蝴蝶结的校服**，可以高度确定这位角色是：

**《Love Live! Sunshine!!》（ラブライブ！サンシャイン!!）中的 津岛 Yoshiko（津島 Yoshiko / つしま よしこ）**。

*   **身份背景：** 她是日本二次元企划《Love Live! Sunshine!!》中偶像团体 **Aqours** 的成员之一。
*   **性格特征：** 在设定中，津岛 Yoshiko 是一个性格非常元气、天真烂漫、有些小恶魔气质的初中生（后升入高中），非常喜爱 pancakes（松饼），经常做出标志性的可爱眨眼、比V字手势并喊出“ Yoshiko的魔法！”等中二又可爱的台词。这幅插画完美展现了她招牌式的甜美与俏皮魅力。

## 2.2.本地图片数据
有时候用户会上传图片数据，而不是图片的url地址。我们需要将图片数据转换成base64字符串，然后发送给模型。

接下来我们会模拟图片上传、转换的过程。

首先，我们安装一个上传组件，用于模拟图片上传。


In [ ]:
!uv add ipywidgets
#在 Jupyter/VSCode Notebook 的 cell 里，**开头带`!`**，IPython 内核会把这一行**丢给系统 bash shell 去执行**，不是 Python 代码。

Resolved 204 packages in 96ms
Checked 198 packages in 775ms


然后，我们创建一个上传组件，用于模拟图片上传。


In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='*', multiple=False)
display(uploader)
# 我上传的是 脑洞学生会 的截图

FileUpload(value=(), accept='*', description='Upload')

In [25]:
print(uploader.value)

({'name': '屏幕截图 2026-10-11 103808.png', 'type': 'image/png', 'size': 718439, 'content': <memory at 0x7374f0be62c0>, 'last_modified': datetime.datetime(2026, 10, 11, 2, 38, 9, tzinfo=datetime.timezone.utc)},)


In [26]:
# 读取图片，转为base64字符串
import base64

# 获取第一个（也是唯一一个）上传的文件
uploaded_file = uploader.value[0]

# 获取其内存视图
content_mv = uploaded_file["content"]

# 转换内存视图->字节
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [27]:
# 组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/png",
    },
    {"type": "text", "text": "给我讲讲图片表达什么"}
])

for chunk, metadata in agent.stream(
    {"messages": [multimodal_question]},
    stream_mode="messages"
):
    print(chunk.content, end="", flush=True)



根据您提供的图片，这是一张典型的日本动漫截图（从画风、人物设定以及左上角的“89人在看”直播/播放弹幕字样可以推断，这很可能是一部正在播出的动漫或视频片段）。

为了准确解析图片所表达的内容，我将从**角色特征、表情肢体语言、画面氛围以及剧情推导**四个维度进行详细的推理，最后给出总结。

---

### 一、 详细推理步骤

#### 1. 角色身份与人物关系识别 (Character Identification)
从人物的外貌和穿着来看，这是一部以日本高中校园为背景的动漫：
*   **右侧的女生（黑发、齐刘海、半睁眼）：** 具有非常典型的“酷女孩”或“女主角”特征。她的半睁眼表情通常用于表现冷静、无奈、淡定，或者面对日常琐事时的无精打采。她穿着深色水手服。
*   **左侧的女生（深色短发/扎发、大眼睛）：** 看起来更加活泼、警觉，眼神中带着一丝好奇或警惕，同样穿着水手服。两人是闺蜜/好友的关系。
*   **中间的男生（戴眼镜、短发）：** 穿着深色男式制服（学兰），表情十分夸张和惊恐。他站在两名女生身后，似乎在保护她们，或者被卷入了某种麻烦中。
*   **作品推测：** 这三人组的外形特征非常符合动漫**《看得见的女孩》（Mieruko-chan / ミエコちゃん）**中的主角 trio：**四谷美子（右，能看见鬼魂的女主）**、**四谷花（左，妹妹/好友）** 和 ** Kyosuke / 涌川（中间戴眼镜的男主/青梅竹马）**。如果是该作，核心设定是“女主能看见鬼魂，但选择无视它们，而朋友看不见，却常被卷入灵异事件”。

#### 2. 表情与肢体语言分析 (Expressions & Body Language)
画面中三人的表情高度紧张，但表现出三种不同层次的心理状态：
*   **右前方的黑发女生（主导者）：**
    *   **动作：** 她伸出一只手指向前方（镜头方向），这个姿势非常具有引导性、指向性，像是在对看不见的对手说“你过来啊”、“就是你，别躲了”，或者在向观众指明某个方向。
    *   **表情：** 尽管眼神有些疲惫和冷淡，但脸上有汗水，说明她内心其实并不像表面看起来那么平静，而是在强装镇定、硬着头皮面对恐惧。
*   **左侧的女生（守护者/观察者）：**
    *   **表情：** 眉毛微蹙，嘴角微微